# Aurora · Verificaciones en Python (V01 a V17)

**Caso:** Tiendas Aurora · **Herramienta:** Python con pandas · **Tiempo:** unos 5 minutos por verificación

Power BI y Python son dos herramientas independientes. Si las dos dan el mismo valor, la cifra es confiable; si no coinciden, una de las dos tiene un error de filtro, de relación, de tipo de dato o de fecha, y hay que encontrarlo antes de seguir.

**Cómo se usa, verificación por verificación**
1. **Prediga** en la celda de texto: escriba su cifra antes de calcular nada.
2. **Construya** la misma cifra en Power BI, con los pasos de la [guía de verificación](verificacion_aurora.md).
3. **Contraste:** ejecute la celda de código de esa verificación.
4. **Compare** con Power BI y con su predicción, y marque ✅ o ❌.
5. **Concluya** en una frase: qué cambia en el control.

**Antes de empezar:** ejecute la celda siguiente una vez (*Run All* o Mayús+Intro). Requiere `pandas`; si falta, ejecute `!pip install pandas`. No modifica ningún archivo.

## Preparación: los datos y las columnas de control
Carga las seis tablas de `datos/demo_aurora/` y crea tres columnas que usan varias verificaciones: `factura_normalizada`, `clave` y `es_duplicado` (las mismas de V03 y V07 en Power Query y en DAX).

In [ ]:
import math

import pandas as pd
from pathlib import Path

# Los datos: en el repositorio clonado (carpeta datos/demo_aurora) o, si trabaja en Colab, directamente desde GitHub
LOCAL = next((p for p in (Path("../datos/demo_aurora"), Path("datos/demo_aurora")) if p.exists()), None)
RUTA = str(LOCAL) if LOCAL else "https://raw.githubusercontent.com/acubillosde/bi-control-organizacional-estudiantes/main/datos/demo_aurora"
print("Datos en:", RUTA)


def leer(nombre, **opciones):
    return pd.read_csv(f"{RUTA}/{nombre}.csv", **opciones)


pagos = leer("pagos", parse_dates=["fecha_factura", "fecha_registro", "fecha_aprobacion", "fecha_pago"])
ventas = leer("ventas_diarias", parse_dates=["fecha"])
prov = leer("proveedores", dtype=str)
emp = leer("empleados", dtype=str)
tiendas = leer("tiendas")
limites = leer("limites_aprobacion")
tablas = {"tiendas": tiendas, "empleados": emp, "limites_aprobacion": limites, "proveedores": prov, "pagos": pagos, "ventas_diarias": ventas}

# Columnas de control (las mismas de V03 y V07 en Power Query y en DAX)
pagos["factura_normalizada"] = pagos.numero_factura.str.replace(r"\D", "", regex=True).str.lstrip("0")
pagos["clave"] = pagos.id_proveedor + "|" + pagos.factura_normalizada + "|" + pagos.valor.astype(str)
pagos["es_duplicado"] = (pagos.id_pago > pagos.groupby("clave").id_pago.transform("min")).astype(int)


def n0(x):
    return f"{x:,.0f}".replace(",", ".")


def n1(x):
    return f"{x:,.1f}".replace(",", "X").replace(".", ",").replace("X", ".")


def n2(x):
    return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

## V01 · Cargue las seis tablas y cuente las filas

**Prediga antes de ejecutar:** ¿Cuántas filas cree que tiene pagos? ¿Y ventas_diarias? Anótelo antes de cargar.

**Su predicción:** ________

In [ ]:
# V01 · Cargue las seis tablas y cuente las filas
for nombre, df in tablas.items():
    print(f"{nombre}: {len(df)} filas")

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V02 · Limpie el maestro sin destruir el original

**Prediga antes de ejecutar:** ¿Cuántas ciudades distintas cree que hay en proveedores? ¿Y después de estandarizar? ¿Y cuántos NIT distintos?

**Su predicción:** ________

In [ ]:
# V02 · Limpie el maestro sin destruir el original
limpia = prov.ciudad.str.lower().str.replace("medellin", "medellín").str.replace("itagui", "itagüí").str.title()
nit_digitos = prov.nit.str.replace(r"\D", "", regex=True)
print("Filas de proveedores:", len(prov))
print("Ciudades distintas antes:", prov.ciudad.nunique())
print("Ciudades distintas después:", limpia.nunique())
print("NIT distintos (solo dígitos):", nit_digitos.nunique())

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V03 · Construya tres columnas de control sobre pagos

**Prediga antes de ejecutar:** De 1.808 pagos, ¿cuántos cree que aprobó la misma persona que los registró? ¿Y cuántos se aprobaron en fin de semana?

**Su predicción:** ________

In [ ]:
# V03 · Construya tres columnas de control sobre pagos
mismo_usuario = (pagos.usuario_registra == pagos.usuario_aprueba).astype(int)
dia_no_habil = (pagos.fecha_aprobacion.dt.dayofweek >= 5).astype(int)   # lunes = 0, sábado = 5
factura_norm = pagos.numero_factura.str.replace(r"\D", "", regex=True).str.lstrip("0")
print("Suma de mismo_usuario:", mismo_usuario.sum())
print("Suma de aprobacion_dia_no_habil:", dia_no_habil.sum())
print("factura_normalizada de FV-2269:", factura_norm[pagos.numero_factura == "FV-2269"].iloc[0])

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V04 · Cree el calendario y encuentre los días sin ventas

**Prediga antes de ejecutar:** La tienda T07 debería vender los 30 días de junio. ¿Cuántos días sin ventas cree que tiene?

**Su predicción:** ________

In [ ]:
# V04 · Cree el calendario y encuentre los días sin ventas
junio = pd.date_range("2025-06-01", "2025-06-30")
vendio = set(ventas[(ventas.id_tienda == "T07") & (ventas.fecha.dt.month == 6)].fecha)
sin_ventas = [d for d in junio if d not in vendio]
print("Días sin ventas de T07 en junio:", len(sin_ventas))
print("Filas esperadas (12 tiendas x 365 días):", 12 * 365, "· filas reales:", len(ventas))

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V05 · Arme la estrella de ventas y pida «ventas por cliente»

**Prediga antes de ejecutar:** ¿Cuánto suman las ventas de 2025? ¿Qué tienda vende más? ¿Puede el modelo decir ventas por cliente?

**Su predicción:** ________

In [ ]:
# V05 · Arme la estrella de ventas y pida «ventas por cliente»
por_tienda = ventas.groupby("id_tienda").ventas.sum().sort_values(ascending=False)
nombre = tiendas.set_index("id_tienda").nombre_tienda[por_tienda.index[0]]
print("Ventas totales 2025: $" + n0(ventas.ventas.sum()), "=", n1(ventas.ventas.sum() / 1e6), "millones")
print("Tienda con más ventas:", por_tienda.index[0], nombre, n1(por_tienda.iloc[0] / 1e6), "millones")
print("Filas de ventas:", len(ventas))

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V06 · Calcule un KPI, un KRI y un KCI

**Prediga antes de ejecutar:** Prediga: ¿cuántos días pasan en promedio entre la factura y el pago? ¿Qué porcentaje de las ventas se devuelve? ¿Qué porcentaje de pagos aprueba quien los registró?

**Su predicción:** ________

In [ ]:
# V06 · Calcule un KPI, un KRI y un KCI
print("KPI · días de factura a pago:", n1((pagos.fecha_pago - pagos.fecha_factura).dt.days.mean()))
print("KRI · devoluciones sobre ventas:", n2(ventas.devoluciones.sum() / ventas.ventas.sum() * 100) + "%")
print("KCI · pagos con el mismo usuario:", n2((pagos.usuario_registra == pagos.usuario_aprueba).mean() * 100) + "%")

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V07 · Reproduzca los tres números de marzo

**Prediga antes de ejecutar:** Marzo de 2025: ¿cuánto sumó «el pago del mes» según la fecha de aprobación? ¿Según la fecha de pago? ¿Y sin duplicados? ¿Por qué no coinciden?

**Su predicción:** ________

In [ ]:
# V07 · Reproduzca los tres números de marzo
marzo_aprobacion = pagos[pagos.fecha_aprobacion.dt.to_period("M") == "2025-03"]
marzo_pago = pagos[pagos.fecha_pago.dt.to_period("M") == "2025-03"]
sin_duplicados = marzo_aprobacion[marzo_aprobacion.es_duplicado == 0]
print("Por fecha de aprobación:", n1(marzo_aprobacion.valor.sum() / 1e6), "millones")
print("Por fecha de pago:", n1(marzo_pago.valor.sum() / 1e6), "millones")
print("Aprobación sin duplicados:", n1(sin_duplicados.valor.sum() / 1e6), "millones")
print("Duplicados en marzo:", len(marzo_aprobacion) - len(sin_duplicados))

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V08 · Mida la calidad del maestro con reglas en DAX

**Prediga antes de ejecutar:** Sobre el maestro SIN limpiar: ¿cuántos proveedores tienen la ciudad mal escrita? ¿Cuántos NIT están repetidos?

**Su predicción:** ________

In [ ]:
# V08 · Mida la calidad del maestro con reglas en DAX
lista = ["Medellín", "Envigado", "Itagüí", "Bello", "Sabaneta"]
nit_digitos = prov.nit.str.replace(r"\D", "", regex=True)
print("Ciudades fuera de lista:", (~prov.ciudad.isin(lista)).sum(), "de", len(prov))   # isin distingue mayúsculas y tildes
print("NIT repetidos:", len(prov) - nit_digitos.nunique())
print("Proveedores con el NIT repetido:", list(prov[nit_digitos.duplicated(keep=False)].id_proveedor))

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V09 · Ponga los umbrales de la matriz en un semáforo

**Prediga antes de ejecutar:** Con los umbrales de la matriz, ¿qué color tendrá cada indicador de Aurora: verde, amarillo o rojo?

**Su predicción:** ________

In [ ]:
# V09 · Ponga los umbrales de la matriz en un semáforo
dias = (pagos.fecha_pago - pagos.fecha_factura).dt.days.mean()
dev = ventas.devoluciones.sum() / ventas.ventas.sum()
sep = ventas[(ventas.id_tienda == "T03") & (ventas.fecha.dt.month == 9)]
dev_t03 = sep.devoluciones.sum() / sep.ventas.sum()
mismo = (pagos.usuario_registra == pagos.usuario_aprueba).mean()
duplicados = pagos.es_duplicado.sum()
semaforo = lambda rojo: "Rojo" if rojo else "Verde"
print("Días de factura a pago:", n1(dias), "→", "Rojo" if dias > 20 else "Amarillo" if dias > 15 else "Verde")
print("Devoluciones (todas las tiendas):", n2(dev * 100) + "%", "→", semaforo(dev > 0.02))
print("Devoluciones de T03 en septiembre:", n2(dev_t03 * 100) + "%", "→", semaforo(dev_t03 > 0.02))
print("Pagos con el mismo usuario:", n2(mismo * 100) + "%", "→", semaforo(mismo > 0))
print("Pagos duplicados:", duplicados, "→", semaforo(duplicados > 0))

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V10 · Haga la torta y compárela con barras

**Prediga antes de ejecutar:** Con una torta de ventas por tienda, ¿puede decir cuál vende más y por cuánto? Pruébelo.

**Su predicción:** ________

In [ ]:
# V10 · Haga la torta y compárela con barras
por_tienda = ventas.groupby("id_tienda").ventas.sum().sort_values(ascending=False)
parte = por_tienda / por_tienda.sum() * 100
print("Tienda con más ventas:", por_tienda.index[0])
print("Participación de las dos primeras:", n1(parte.iloc[0]) + "% y " + n1(parte.iloc[1]) + "%")
print("Diferencia entre las dos primeras:", n1((por_tienda.iloc[0] - por_tienda.iloc[1]) / 1e6), "millones")

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V11 · Resalte las devoluciones de T03 frente al resto

**Prediga antes de ejecutar:** ¿En qué mes devolvió T03 mucho más que el resto de tiendas? ¿Cuánto fue?

**Su predicción:** ________

In [ ]:
# V11 · Resalte las devoluciones de T03 frente al resto
ventas["mes"] = ventas.fecha.dt.month
def pct(df):
    return df.groupby("mes")[["devoluciones", "ventas"]].sum().pipe(lambda g: g.devoluciones / g.ventas * 100)
t03, resto = pct(ventas[ventas.id_tienda == "T03"]), pct(ventas[ventas.id_tienda != "T03"])
print(pd.DataFrame({"T03 %": t03.round(2), "Resto %": resto.round(2)}))
print("Mes del pico de T03:", t03.idxmax(), "·", n2(t03.max()) + "%", "· resto ese mes:", n2(resto[t03.idxmax()]) + "%")

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V12 · Haga un titular dinámico y póngalo a prueba

**Prediga antes de ejecutar:** Si el titular dice «T03 devolvió X% de sus ventas», ¿qué dirá en enero? ¿Sigue siendo un buen titular?

**Su predicción:** ________

In [ ]:
# V12 · Haga un titular dinámico y póngalo a prueba
ventas["mes"] = ventas.fecha.dt.month
def pct(df, mes):
    g = df[df.mes == mes]
    return g.devoluciones.sum() / g.ventas.sum() * 100
for mes, nombre in [(9, "septiembre"), (1, "enero")]:
    t03, resto = pct(ventas[ventas.id_tienda == "T03"], mes), pct(ventas[ventas.id_tienda != "T03"], mes)
    print(f"Titular en {nombre}: T03 devolvió {n1(t03)}% de sus ventas; el resto de tiendas, {n1(resto)}%")
    if t03 > 2:
        print(f"Titular mejorado en {nombre}: Alerta: T03 devolvió {n1(t03)}% de sus ventas, {round(t03 / resto)} veces el resto")

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V13 · Pruebe Benford sobre el valor de los pagos

**Prediga antes de ejecutar:** ¿Con qué frecuencia cree que el valor de un pago empieza por 1? Prediga el porcentaje para todos los pagos y para el usuario aur006.

**Su predicción:** ________

In [ ]:
# V13 · Pruebe Benford sobre el valor de los pagos
primer = pagos.valor.astype(str).str[0].astype(int)
uno = lambda mascara: (primer[mascara] == 1).mean() * 100   # % de pagos del grupo cuyo valor empieza por 1
print("Esperado para el dígito 1:", n1(math.log10(2) * 100) + "%")
print("Todos los pagos:", n1((primer == 1).mean() * 100) + "%")
print("Resto de usuarios:", n1(uno(pagos.usuario_registra != "aur006")) + "%")
print("aur006:", n1(uno(pagos.usuario_registra == "aur006")) + "%", "·", (pagos.usuario_registra == "aur006").sum(), "pagos")

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V14 · Cuente duplicados exactos y aproximados

**Prediga antes de ejecutar:** ¿Cuántos pagos están repetidos con mismo proveedor, factura y valor? ¿Y cuántos más aparecen si se normaliza el número de factura?

**Su predicción:** ________

In [ ]:
# V14 · Cuente duplicados exactos y aproximados
exactos = pagos.duplicated(["id_proveedor", "numero_factura", "valor"]).sum()
normalizados = pagos.es_duplicado.sum()
print("Duplicados exactos:", exactos)
print("Duplicados con la factura normalizada:", normalizados)
print("Duplicados aproximados:", normalizados - exactos)

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V15 · Busque pagos pegados al límite de aprobación

**Prediga antes de ejecutar:** El jefe de compras aprueba hasta $8 millones. ¿Cuántos pagos del año caen entre el 95% y el 100% del límite? ¿Y cuántos son del mismo proveedor en agosto?

**Su predicción:** ________

In [ ]:
# V15 · Busque pagos pegados al límite de aprobación
cerca = pagos[(pagos.valor >= 0.95 * 8_000_000) & (pagos.valor < 8_000_000)]
print("Pagos entre 95% y 100% del límite (año):", len(cerca))
print(cerca.id_proveedor.value_counts().head(3))
ago = cerca[(cerca.id_proveedor == "P017") & (cerca.fecha_aprobacion.dt.month == 8)]
print("P017 en agosto:", len(ago), "pagos entre", n2(ago.valor.min() / 1e6), "y", n2(ago.valor.max() / 1e6), "millones")

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V16 · Mida el valor y la concentración de las dos señales

**Prediga antes de ejecutar:** Los pagos aprobados por quien los registró: ¿de cuántos usuarios son y cuánto valen? Y los de fin de semana: ¿sábado o domingo?

**Su predicción:** ________

In [ ]:
# V16 · Mida el valor y la concentración de las dos señales
mismo = pagos[pagos.usuario_registra == pagos.usuario_aprueba]
finde = pagos[pagos.fecha_aprobacion.dt.dayofweek >= 5]
print("Pagos del mismo usuario:", len(mismo), "·", n1(mismo.valor.sum() / 1e6), "millones · usuarios:", list(mismo.usuario_registra.unique()))
print("Pagos en día no hábil:", len(finde), "· sábados:", (finde.fecha_aprobacion.dt.dayofweek == 5).sum(), "· domingos:", (finde.fecha_aprobacion.dt.dayofweek == 6).sum())
print("Valor de los pagos de fin de semana:", n1(finde.valor.sum() / 1e6), "millones")

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## V17 · Cruce cuentas bancarias de proveedores y de nómina

**Prediga antes de ejecutar:** ¿Cuántos proveedores comparten cuenta bancaria con la nómina de un empleado? ¿Qué patrón tienen sus pagos?

**Su predicción:** ________

In [ ]:
# V17 · Cruce cuentas bancarias de proveedores y de nómina
coinciden = prov[prov.cuenta_bancaria.isin(emp.cuenta_nomina)]
print("Proveedores con cuenta de nómina:", len(coinciden), list(coinciden.id_proveedor))
pr = pagos[pagos.id_proveedor.isin(coinciden.id_proveedor)]
print("Pagos a ese proveedor:", len(pr), "·", n1(pr.valor.sum() / 1e6), "millones")
print("Proveedor creado:", coinciden.fecha_creacion.iloc[0], "· primer pago aprobado:", pr.fecha_aprobacion.min().date(), "· último:", pr.fecha_aprobacion.max().date())
print("Quién registró los pagos:", list(pr.usuario_registra.unique()), "· valor máximo:", n1(pr.valor.max() / 1e6), "millones")

**¿Coincide con Power BI y con su predicción?** ✅ / ❌

**Una frase: ¿qué cambia en el control?** ________

## Cierre
Si alguna verificación marcó ❌, vuelva a la guía [verificacion_aurora.md](verificacion_aurora.md): cada una trae «Si no coincide» con las causas más frecuentes. Si Power BI y Python difieren, una de las dos tiene el error.